In [1]:
import xml.etree.ElementTree as ET
import copy

In [ ]:
persontrips_filepath = "persontrips_maloja_school_modded.xml"
persontrips_filepath_output = "persontrips_maloja_school_modded_with_parents.xml"
routes_filepath = "persontrips_maloja_school_resolved_offseason_bus_modded.rou.xml"
routes_filepath_output = "persontrips_maloja_school_resolved_offseason_bus_modded_with_parents.rou.xml"

In [3]:
guardians = {
    "taz_bregaglia_kinderkarten_elementary_school_maloja": 3,
    "taz_bregaglia_kinderkarten_elementary_school_vicosoprano": 3,
    "taz_sils_im_engadin_kinderkarten_elementary_school": 1,
    "taz_silvaplana_kinderkarten_elementary_school": 1,
    "taz_st_moritz_kinderkarten_elementary_school_dorf": 6,
    "taz_st_moritz_kinderkarten_elementary_school_secondary_school_grevas": 9
}
guardians

{'taz_bregaglia_kinderkarten_elementary_school_maloja': 3,
 'taz_bregaglia_kinderkarten_elementary_school_vicosoprano': 3,
 'taz_sils_im_engadin_kinderkarten_elementary_school': 1,
 'taz_silvaplana_kinderkarten_elementary_school': 1,
 'taz_st_moritz_kinderkarten_elementary_school_dorf': 6,
 'taz_st_moritz_kinderkarten_elementary_school_secondary_school_grevas': 9}

In [4]:
guardians_to_schools = copy.deepcopy(guardians)
guardians_from_schools = copy.deepcopy(guardians)

In [5]:
persontrips_file_xmltree = ET.parse(persontrips_filepath)
persontrips_file_xmlroot = persontrips_file_xmltree.getroot()

In [ ]:
routes_file_xmltree = ET.parse(routes_filepath)
routes_file_xmlroot = routes_file_xmltree.getroot()

In [7]:
count_to_school = 0
count_from_school = 0

for person_child in list(routes_file_xmlroot):
    if "p" not in str(person_child.get("id")):
        person_child.set("id", "s" + str(person_child.get("id")))

for person_child in list(persontrips_file_xmlroot):
    if "p" not in str(person_child.get("id")):
        person_child.set("id", "s" + str(person_child.get("id")))
        for persontrip_child in person_child:
            try:
                if guardians_to_schools[str(persontrip_child.get("toTaz"))] > 0:
                    parent = copy.deepcopy(person_child)
                    parent.set("id", (str(parent.get("id"))).replace("s", "p"))
                    persontrips_file_xmlroot.insert(list(persontrips_file_xmlroot).index(person_child), parent)
                    guardians_to_schools[str(persontrip_child.get("toTaz"))] = guardians_to_schools[str(persontrip_child.get("toTaz"))] - 1
                    for route in list(routes_file_xmlroot):
                        if str(route.get("id")) == str(person_child.get("id")):
                            p_route = copy.deepcopy(route)
                            p_route.set("id", (str(p_route.get("id"))).replace("s", "p"))
                            routes_file_xmlroot.insert(list(routes_file_xmlroot).index(route), p_route)
                    count_to_school += 1
            except KeyError:
                continue

for person_child in list(persontrips_file_xmlroot):
    if "p" not in str(person_child.get("id")):
        for persontrip_child in person_child:
            try:
                if guardians_from_schools[str(persontrip_child.get("fromTaz"))] > 0:
                    parent = copy.deepcopy(person_child)
                    parent.set("id", (str(parent.get("id"))).replace("s", "p"))
                    persontrips_file_xmlroot.insert(list(persontrips_file_xmlroot).index(person_child), parent)
                    guardians_from_schools[str(persontrip_child.get("fromTaz"))] = guardians_from_schools[str(persontrip_child.get("fromTaz"))] - 1
                    for route in list(routes_file_xmlroot):
                        if str(route.get("id")) == str(person_child.get("id")):
                            p_route = copy.deepcopy(route)
                            p_route.set("id", (str(p_route.get("id"))).replace("s", "p"))
                            routes_file_xmlroot.insert(list(routes_file_xmlroot).index(route), p_route)
                    count_from_school += 1
            except KeyError:
                continue

print("parents to school: " + str(count_to_school) + " parents from school: " + str(count_from_school))

parents to school: 23 parents from school: 23


In [ ]:
persontrips_file_xmltree.write(persontrips_filepath_output)
routes_file_xmltree.write(routes_filepath_output)